# CIS 6211 – Foundations of Data Science
## Lab 5: Linear Algebra

**Student Name:** Rana Sultan Alhinidy  
**Course:** CIS 6211 | King Khalid University


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
from sklearn.decomposition import PCA
%matplotlib inline

### 📌 Cell Explanation
This cell imports the required libraries:
- **numpy** is used for numerical operations and generating random data.
- **matplotlib** is used for 2D and 3D plotting.
- **FancyArrowPatch** and **proj3d** are used to draw 3D arrows representing coordinate axes.
- **PCA from sklearn** is used to perform Principal Component Analysis for dimensionality reduction.

## Cell 1: Arrow3D Helper Class

In [ ]:
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return np.min(zs)

### 📌 Cell Explanation
This cell defines a custom class **Arrow3D** that extends matplotlib's `FancyArrowPatch` to draw arrows in 3D space.

It takes 3D coordinates (x, y, z) and projects them onto the 2D screen using `proj3d.proj_transform`. This class is used throughout the lab to draw the three coordinate axes (x, y, z) in the 3D plots.

## Cell 2: Random Points in 3D Space

In [ ]:
d = 1.25
n = 10
x = np.random.uniform(-d, d, n)
y = np.random.uniform(-d, d, n)
z = np.random.uniform(-d, d, n)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_aspect('equal')
ax.plot(x, y, z, 'bo', markersize=7)

m = 2.5
xa = Arrow3D([-m, m], [0, 0], [0, 0], mutation_scale=20, lw=1, linestyle=":", arrowstyle="<|-|>", color='gray')
ya = Arrow3D([0, 0], [-m, m], [0, 0], mutation_scale=20, lw=1, linestyle=":", arrowstyle="<|-|>", color='gray')
za = Arrow3D([0, 0], [0, 0], [-m, m], mutation_scale=20, lw=1, linestyle=":", arrowstyle="<|-|>", color='gray')
ax.add_artist(xa)
ax.add_artist(ya)
ax.add_artist(za)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_zlim(-1.5, 1.5)
ax.set_axis_off()
plt.show()

### 📌 Cell Explanation
This cell generates **10 random 3D points** uniformly distributed within the range [-1.25, 1.25] for each axis.

The points are plotted as blue dots in a 3D coordinate system with gray dashed arrows showing the three axes (x, y, z). This represents raw data points in a three-dimensional feature space before any transformation.

In data science, every data point with multiple features can be thought of as a point in high-dimensional space.

## Cell 3: Vectors from Origin with Sphere

In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111, projection='3d')
ax.set_box_aspect([1, 1, 1])

for i in range(0,n):
    ax.plot([0,x[i]],[0,y[i]], [0,z[i]], color='b')
ax.plot(x, y, z, 'bo', markersize=7)

u = np.linspace(0, 2 * np.pi, 500)
v = np.linspace(0, np.pi, 500)
x1 = 1 * np.outer(np.cos(u), np.sin(v))
y1 = 1 * np.outer(np.sin(u), np.sin(v))
z1 = 1 * np.outer(np.ones(np.size(u)), np.cos(v))
ax.plot_surface(x1, y1, z1, color='y', alpha=0.2, linewidth=0)
ax.set_axis_off()

### 📌 Cell Explanation
This cell takes the same random points and draws **lines (vectors) from the origin (0,0,0)** to each point, showing that every data point can be represented as a vector.

A transparent **yellow sphere of radius 1** is added. The vectors have different lengths (magnitudes): some extend beyond the sphere while others remain inside it.

This visualizes a key concept: **vectors have both direction and magnitude**. In machine learning, feature vectors represent data points as arrows in space.

## Cell 4: Unit Vectors

In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111, projection='3d')
ax.set_box_aspect([1, 1, 1])

for i in range(0,n):
    mag = (x[i]**2 + y[i]**2 + z[i]**2)**0.5
    ax.plot([0,x[i]/mag],[0,y[i]/mag], [0,z[i]/mag], color='r', marker='o', markersize=7)

u = np.linspace(0, 2 * np.pi, 500)
v = np.linspace(0, np.pi, 500)
x1 = np.outer(np.cos(u), np.sin(v))
y1 = np.outer(np.sin(u), np.sin(v))
z1 = np.outer(np.ones(np.size(u)), np.cos(v))
ax.plot_surface(x1, y1, z1, color='y', alpha=0.2, linewidth=0)
ax.set_axis_off()

### 📌 Cell Explanation
This cell **normalizes each vector to a length of exactly 1**, creating unit vectors.

The magnitude is calculated using the **Euclidean distance formula**: `mag = √(x² + y² + z²)`, then each component is divided by the magnitude.

All red vectors now end exactly on the surface of the unit sphere. This demonstrates **vector normalization**, which is important in data science for:
- Removing the effect of scale while preserving direction
- Computing **cosine similarity** in text analysis
- **Feature normalization** before training models

## Cell 5: PCA Visualization

In [ ]:
numPoints = 50
x = np.random.uniform(3,12,numPoints) + 3
y = x + np.random.uniform(-3,3,numPoints) + 3

X = np.array([[xi,yi] for xi,yi in zip(x,y)])
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X)

pts = [abs(d[1]) for d in pca_result]
min_index = np.argmin(pts)

x_pca = np.linspace(0,18,30)
r = np.matmul([-5,0],pca.components_)
centroid_orig = [sum(x)/float(len(x)), sum(y)/float(len(x))]
centroid_pca = [sum(pca_result[:,0])/float(len(x)), sum(pca_result[:,1])/float(len(x))]

plt.plot(x,y,'ko', label='orig points', markersize=4)
plt.plot(pca_result[:,0], pca_result[:,1],'ro', label='pca 2d points', markersize=4)
plt.plot([centroid_orig[0], centroid_pca[0]], [centroid_orig[1], centroid_pca[1]],'bo', label='centroids')
plt.plot([-9,9],[0,0],'k-', label='PCA axis')

line_xrange = np.linspace(5,16,20)
line_slope = (centroid_orig[1] - y[min_index])/(centroid_orig[0] - x[min_index])
b = -centroid_orig[0]*line_slope + centroid_orig[1]
plt.plot(line_xrange, line_slope*line_xrange + b, 'k-')
plt.legend(loc='upper left')
plt.xlim(-10,17)
plt.ylim(-3,22)

### 📌 Cell Explanation
This cell demonstrates **Principal Component Analysis (PCA)**:

1. 50 random 2D points with a linear trend are generated.
2. PCA is applied to find the **principal components** — the directions of maximum variance.
3. The plot shows:
   - **Black dots** = original points
   - **Red dots** = PCA-transformed points
   - **Blue dots** = centroids (center of mass)
   - **Black line** = first principal component (direction of greatest variance)

PCA essentially **rotates the coordinate system** so that the first axis captures the most variation in the data. This is fundamental for dimensionality reduction.

## Cell 6: PCA Projection

In [ ]:
plt.figure(figsize=(5,5))
plt.plot(x,y,'ko', markersize=4)
plt.plot(line_xrange, line_slope*line_xrange + b, 'k-')
for xi,yi in zip(x,y):
    bi = 1/line_slope * xi + yi
    new_x = (bi-b)/(line_slope+(1/line_slope))
    new_y = line_slope*new_x + b
    plt.plot([new_x], [new_y], 'ro', markersize=3.5)
    plt.plot([xi, new_x], [yi, new_y], 'k--')
plt.xlim(3,18)
plt.ylim(6,21)

### 📌 Cell Explanation
This cell shows the **geometric interpretation of PCA projection**:

- Each original point (black dot) is projected **perpendicularly** onto the principal component line.
- The dashed lines show the perpendicular distance from each point to the line.
- The red dots show where each point lands on the line.

This is **dimensionality reduction**: the 2D data is compressed to 1D by keeping only the position along the principal component.

The key idea: PCA **minimizes the total perpendicular projection error** from points to the line, preserving as much variance as possible.